# Generador de plan semanal (ciclo completo)

In [52]:
from pathlib import Path
import pandas as pd, numpy as np, math
from datetime import datetime

def ceil2p5(x):
    if pd.isna(x): return np.nan
    return math.ceil(x/2.5)*2.5

BASE = Path("/Users/bullville/Documents/GitHub/bullville_M1/gym/plantillas")
porc = pd.read_csv(BASE / "porcentajes.csv", index_col=0)
porc.index = porc.index.str.strip()
porc.columns = porc.columns.map(str).str.strip()   # "1".."15"

reps = pd.read_csv(BASE / "repeticiones.csv", index_col=0)
reps.index = reps.index.str.strip()
reps.columns = reps.columns.map(str).str.strip()   # "1".."15"

rm   = pd.read_csv(BASE / "rm_alemania.csv", index_col=0)

weeks = [str(i) for i in range(1, 16)]


In [53]:
# 1) Actualiza RM
today = datetime.now().date().isoformat()
last_col = rm.columns[-1]

extras = ["roller", "muscle_up"]

# Trabajar en float para evitar FutureWarning
base = rm[last_col].astype("float64")

# +5 si es extra, +2.5 en el resto
mask_extra = rm.index.isin(extras)
new_vals = base + np.where(mask_extra, 5.0, 2.5)

rm[today] = new_vals
rm.to_csv(BASE / "rm_alemania.csv")
rm_latest = rm[today]


In [54]:
print(rm)

               2023-05-15  2025-08-11
ejercicio                            
clean_jerk          100.0       102.5
snatch               70.0        72.5
squad               100.0       102.5
pull_up             110.0       112.5
press_militar        70.0        72.5
press_banca         100.0       102.5
deadlift            140.0       142.5
gluteo_bridge       150.0       152.5
curl_nordico         60.0        62.5
roller                5.0        10.0
muscle_up             3.0         8.0


In [55]:
# --- 2) PLAN ---
rows = [
    "clean_jerk_1","clean_jerk_2","clean_jerk_3","clean_jerk_4",
    "snatch_1","snatch_2","snatch_3","snatch_4",
    "deadlift_1","deadlift_2","deadlift_3","deadlift_4",
    "olimpico_rep_1","olimpico_rep_2","olimpico_rep_3","olimpico_rep_4",
    "squad","pull_up","press_militar","press_banca","gluteo_bridge","curl_nordico",
    "no_olimpico_rep","roller","muscle_up"
]
plan = pd.DataFrame(index=rows, columns=weeks, dtype=float)

def ceil2p5(x):
    if pd.isna(x): return np.nan
    return math.ceil(x/2.5)*2.5

# Olímpicos (pesos) -> RM * % de olimpico_1..4 (o clean_jerk_1..4 si tuvieras ese esquema)
for i in [1, 2, 3, 4]:
    pct_row = f"olimpico_{i}" if f"olimpico_{i}" in porc.index else f"clean_jerk_{i}"
    if pct_row not in porc.index:
        raise KeyError(f"Falta la fila de porcentajes para la sesión {i} ('{pct_row}').")
    pcts = porc.loc[pct_row, weeks].astype(float)

    for lift in ["clean_jerk", "snatch", "deadlift"]:
        base_rm = rm_latest.get(lift, np.nan)
        plan.loc[f"{lift}_{i}", weeks] = (base_rm * pcts).apply(ceil2p5).values

# Reps olímpicos -> coger DIRECTO de repeticiones.csv (índice = 'semana')
for i in [1, 2, 3, 4]:
    for candidate in (f"olimpico_{i}_rep", f"clean_jerk_{i}_rep"):
        if candidate in reps.index:
            plan.loc[f"olimpico_rep_{i}", weeks] = reps.loc[candidate, weeks].astype("Int64").values
            break
    else:
        # Si no existe la fila en el CSV, deja NaN (o lanza error si prefieres)
        plan.loc[f"olimpico_rep_{i}", weeks] = pd.Series([pd.NA]*len(weeks), index=weeks, dtype="Int64").values

# No olímpicos (pesos) -> usar fila 'no_olimpico' de porcentajes
pct_no = porc.loc["no_olimpico", weeks].astype(float) if "no_olimpico" in porc.index else pd.Series(1.0, index=weeks)
for lift in ["squad","pull_up","press_militar","press_banca","gluteo_bridge","curl_nordico"]:
    base_rm = rm_latest.get(lift, np.nan)
    plan.loc[lift, weeks] = (base_rm * pct_no).apply(ceil2p5).values

# Reps no olímpicos -> UNA sola fila 'no_olimpico_rep'
if "no_olimpico_rep" in reps.index:
    plan.loc["no_olimpico_rep", weeks] = reps.loc["no_olimpico_rep", weeks].astype("Int64").values
else:
    plan.loc["no_olimpico_rep", weeks] = pd.Series([pd.NA]*len(weeks), index=weeks, dtype="Int64").values

# Extras -> peso = RM + sumador semanal de reps (si no hay fila, suma 0)
for extra in ["roller", "muscle_up"]:
    base_rm = rm_latest.get(extra, np.nan)
    add = reps.loc[extra, weeks].astype(float) if extra in reps.index else pd.Series(0.0, index=weeks)
    plan.loc[extra, weeks] = (pd.Series(base_rm, index=weeks, dtype=float) + add).values

# Cast de filas de repeticiones a enteros
for r in [f"olimpico_rep_{i}" for i in [1,2,3,4]] + ["no_olimpico_rep"]:
    plan.loc[r] = plan.loc[r].astype("Int64")

print(plan)

                     1      2      3      4      5      6      7      8  \
clean_jerk_1      70.0   70.0   80.0   80.0   80.0   85.0   85.0   85.0   
clean_jerk_2      77.5   77.5   80.0   87.5   87.5   87.5   87.5   87.5   
clean_jerk_3      80.0   80.0   87.5   90.0   90.0   90.0   92.5   92.5   
clean_jerk_4      85.0   87.5   92.5   92.5   97.5   97.5   97.5  100.0   
snatch_1          50.0   50.0   57.5   57.5   57.5   60.0   60.0   60.0   
snatch_2          55.0   55.0   57.5   62.5   62.5   62.5   62.5   62.5   
snatch_3          57.5   57.5   62.5   65.0   65.0   65.0   67.5   67.5   
snatch_4          60.0   62.5   67.5   67.5   70.0   70.0   70.0   72.5   
deadlift_1        97.5   97.5  110.0  110.0  110.0  117.5  117.5  117.5   
deadlift_2       107.5  107.5  110.0  120.0  120.0  120.0  120.0  120.0   
deadlift_3       110.0  110.0  120.0  125.0  125.0  125.0  130.0  130.0   
deadlift_4       117.5  120.0  130.0  130.0  135.0  135.0  135.0  140.0   
olimpico_rep_1     5.0   

In [56]:
# Verificación básica
if "plan" not in globals():
    raise RuntimeError("No se encontró el DataFrame 'plan' en memoria. Ejecuta antes la celda que lo construye.")

# Columnas (semanas)
weeks = [str(c) for c in plan.columns]

# Filas que pide el enunciado
# Nota: en el mensaje original aparecía 'clean_jerl' (typo); aquí usamos 'clean_jerk'
rows_A = (
    [f"clean_jerk_{i}" for i in [1,2,3,4]] +
    [f"snatch_{i}" for i in [1,2,3,4]] +
    ["squad", "pull_up", "press_militar", "press_banca"] +
    [f"olimpico_rep_{i}" for i in [1,2,3,4]] + ["no_olimpico_rep"]
)

rows_B = (
    [f"deadlift_{i}" for i in [1,2,3,4]] +
    ["gluteo_bridge", "muscle_up", "curl_nordico", "roller"] +
    [f"olimpico_rep_{i}" for i in [1,2,3,4]] + ["no_olimpico_rep"]
)

# Filtrar solo las filas que existan en 'plan' (por si acaso)
rows_A_present = [r for r in rows_A if r in plan.index]
rows_B_present = [r for r in rows_B if r in plan.index]

plan_A = plan.loc[rows_A_present, weeks].copy()
plan_B = plan.loc[rows_B_present, weeks].copy()

In [57]:
# Carpeta de salida (intenta relativa al proyecto; si no, usa /mnt/data)
root = Path("/Users/bullville/Documents/GitHub/bullville_M1/gym")
planes_dir = (root / "planes")
planes_dir.mkdir(parents=True, exist_ok=True)

# Nombre del archivo por fecha
fname = f"{datetime.now().date().isoformat()}.xlsx"
excel_path = planes_dir / fname

# Guardar en dos pestañas
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    plan_A.to_excel(writer, sheet_name="A")
    plan_B.to_excel(writer, sheet_name="B")

print(f"Archivo guardado en: {excel_path}")
print(f"Hoja A: {plan_A.shape[0]} filas x {plan_A.shape[1]} semanas")
print(f"Hoja B: {plan_B.shape[0]} filas x {plan_B.shape[1]} semanas")

Archivo guardado en: /Users/bullville/Documents/GitHub/bullville_M1/gym/planes/2025-08-11.xlsx
Hoja A: 17 filas x 15 semanas
Hoja B: 13 filas x 15 semanas
